In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
df_sales = pd.read_csv('../data/raw/retail_sales_data.csv')
df_sales.head()

,Order_ID,Order_Date,Ship_Date,Customer_ID,Customer_Name,Segment,Region,Product_ID,Product_Category,Product_Sub_Category,Product_Name,Sales,Quantity,Discount,Profit,Shipping_Cost,Order_Priority,Unit_Price,Revenue
0,ORD000001,2022-01-01 00:00:00,2022-01-02 00:00:00,CUST1127,Customer_1275,Consumer,Central,PROD0215,Office Supplies,Appliances,Product_2,235.926440,1,0.00,39.775401,8.157251,High,235.926440,235.926440
1,ORD000002,2022-01-01 01:00:00,2022-01-02 01:00:00,CUST1460,Customer_1334,Corporate,East,PROD0002,Furniture,Shirts,Product_206,180.502739,9,0.15,12.835350,3.822394,Medium,20.055860,180.502739
2,ORD000003,2022-01-01 02:00:00,2022-01-02 02:00:00,CUST0861,Customer_1744,Corporate,East,PROD0121,Electronics,Paper,Product_230,40.976998,7,0.00,46.203636,9.084032,Low,5.853857,40.976998
3,ORD000004,2022-01-01 03:00:00,2022-01-02 03:00:00,CUST1295,Customer_833,Corporate,East,PROD0103,Clothing,Appliances,Product_335,421.665820,9,0.10,14.270852,6.348036,High,46.851758,421.665820
4,ORD000005,2022-01-01 04:00:00,2022-01-02 04:00:00,CUST1131,Customer_140,Consumer,South,PROD0149,Office Supplies,Accessories,Product_192,151.938194,5,0.00,-62.712565,36.467989,High,30.387639,151.938194


In [3]:
print("Shape:", df_sales.shape)
print(df_sales.columns.tolist())

Shape: (10000, 19)
['Order_ID', 'Order_Date', 'Ship_Date', 'Customer_ID', 'Customer_Name', 'Segment', 'Region', 'Product_ID', 'Product_Category', 'Product_Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Shipping_Cost', 'Order_Priority', 'Unit_Price', 'Revenue']


In [4]:
df_sales.dtypes

Order_ID                 object
Order_Date               object
Ship_Date                object
Customer_ID              object
Customer_Name            object
Segment                  object
Region                   object
Product_ID               object
Product_Category         object
Product_Sub_Category     object
Product_Name             object
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
Shipping_Cost           float64
Order_Priority           object
Unit_Price              float64
Revenue                 float64
dtype: object

In [5]:
print("Missing values per column:")
print(df_sales.isnull().sum())
print()
print("Duplicate rows:", df_sales.duplicated().sum())

Missing values per column:
Order_ID                 0
Order_Date               0
Ship_Date                0
Customer_ID              0
Customer_Name           50
Segment                  0
Region                   0
Product_ID               0
Product_Category         0
Product_Sub_Category     0
Product_Name             0
Sales                    0
Quantity                 0
Discount                 0
Profit                  30
Shipping_Cost            0
Order_Priority           0
Unit_Price               0
Revenue                  0
dtype: int64

Duplicate rows: 0


In [6]:
df_sales.describe()

,Sales,Quantity,Discount,Profit,Shipping_Cost,Unit_Price,Revenue
count,10000.000000,10000.000000,10000.00000,9970.000000,10000.000000,10000.000000,10000.000000
mean,109.919718,5.006500,0.09154,19.806369,6.524051,34.595699,109.919718
std,71.361990,2.575679,0.08378,30.514838,3.686363,42.690397,71.361990
min,10.353156,1.000000,0.00000,-97.441172,2.007435,1.363330,10.353156
25%,58.329671,3.000000,0.00000,-0.676147,3.849082,11.106737,58.329671
50%,92.921765,5.000000,0.10000,20.329955,5.594567,20.529712,92.921765
75%,145.130765,7.000000,0.15000,40.305723,8.157661,39.758830,145.130765
max,622.935663,9.000000,0.25000,133.699236,36.467989,540.180333,622.935663


In [7]:
df_sales['Profit'] = df_sales['Profit'].fillna(df_sales['Profit'].median())
df_sales['Customer_Name'] = df_sales['Customer_Name'].fillna('Unknown')

In [8]:
print(df_sales.isnull().sum().sum())

0


In [9]:
df_sales['Order_Date'] = pd.to_datetime(df_sales['Order_Date'], errors='coerce')
df_sales['Ship_Date'] = pd.to_datetime(df_sales['Ship_Date'], errors='coerce')

In [10]:
print(df_sales[['Order_Date', 'Ship_Date']].dtypes)
print("New missing dates:", df_sales['Order_Date'].isnull().sum() + df_sales['Ship_Date'].isnull().sum())

Order_Date    datetime64[ns]
Ship_Date     datetime64[ns]
dtype: object
New missing dates: 0


In [11]:
categorical_cols = ['Segment', 'Region', 'Product_Category', 'Product_Sub_Category', 'Order_Priority']
for col in categorical_cols:
    df_sales[col] = df_sales[col].astype('category')

print(df_sales[categorical_cols].dtypes)

Segment                 category
Region                  category
Product_Category        category
Product_Sub_Category    category
Order_Priority          category
dtype: object


In [12]:
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

outliers_sales, lb, ub = detect_outliers_iqr(df_sales, 'Sales')
print(f"Outliers in Sales: {len(outliers_sales)}")
print(f"Acceptable range: {lb:.2f} to {ub:.2f}")

Outliers in Sales: 328
Acceptable range: -71.87 to 275.33


In [13]:
df_sales['Sales_Capped'] = df_sales['Sales'].clip(lower=lb, upper=ub)

print("Before capping - max Sales:", df_sales['Sales'].max())
print("After capping - max Sales:", df_sales['Sales_Capped'].max())
print("Rows changed:", (df_sales['Sales'] != df_sales['Sales_Capped']).sum())

Before capping - max Sales: 622.9356626613855
After capping - max Sales: 275.33240624746435
Rows changed: 328


In [14]:
outliers_profit, lb_p, ub_p = detect_outliers_iqr(df_sales, 'Profit')
df_sales['Profit_Capped'] = df_sales['Profit'].clip(lower=lb_p, upper=ub_p)

print(f"Outliers in Profit: {len(outliers_profit)}")
print(f"Acceptable range: {lb_p:.2f} to {ub_p:.2f}")
print("Rows changed:", (df_sales['Profit'] != df_sales['Profit_Capped']).sum())

Outliers in Profit: 87
Acceptable range: -61.87 to 101.47
Rows changed: 87


In [15]:
df_sales['Year'] = df_sales['Order_Date'].dt.year
df_sales['Month'] = df_sales['Order_Date'].dt.month
df_sales['Quarter'] = df_sales['Order_Date'].dt.quarter
df_sales['Day_of_Week'] = df_sales['Order_Date'].dt.dayofweek
df_sales['Is_Weekend'] = df_sales['Day_of_Week'].isin([5, 6]).astype(int)
df_sales['Delivery_Days'] = (df_sales['Ship_Date'] - df_sales['Order_Date']).dt.days
df_sales['Profit_Margin_Pct'] = (df_sales['Profit'] / df_sales['Sales']) * 100

df_sales[['Order_Date', 'Year', 'Month', 'Quarter', 'Day_of_Week', 'Is_Weekend', 'Delivery_Days', 'Profit_Margin_Pct']].head()

,Order_Date,Year,Month,Quarter,Day_of_Week,Is_Weekend,Delivery_Days,Profit_Margin_Pct
0,2022-01-01 00:00:00,2022,1,1,5,1,1,16.859239
1,2022-01-01 01:00:00,2022,1,1,5,1,1,7.110889
2,2022-01-01 02:00:00,2022,1,1,5,1,1,112.755054
3,2022-01-01 03:00:00,2022,1,1,5,1,1,3.384399
4,2022-01-01 04:00:00,2022,1,1,5,1,1,-41.275049


In [16]:
print("Negative delivery days (Ship before Order):", (df_sales['Delivery_Days'] < 0).sum())
print("Delivery days range:", df_sales['Delivery_Days'].min(), "to", df_sales['Delivery_Days'].max())
print()
print("Profit margin range:", df_sales['Profit_Margin_Pct'].min(), "to", df_sales['Profit_Margin_Pct'].max())
print("Rows with margin > 100%:", (df_sales['Profit_Margin_Pct'] > 100).sum())

Negative delivery days (Ship before Order): 0
Delivery days range: 1 to 1

Profit margin range: -335.74670645992535 to 573.6327437537788
Rows with margin > 100%: 779


In [17]:
df_sales['Profit_Margin_Pct_Capped'] = (df_sales['Profit_Capped'] / df_sales['Sales_Capped']) * 100

print("Original margin range:", df_sales['Profit_Margin_Pct'].min(), "to", df_sales['Profit_Margin_Pct'].max())
print("Capped margin range:", df_sales['Profit_Margin_Pct_Capped'].min(), "to", df_sales['Profit_Margin_Pct_Capped'].max())

Original margin range: -335.74670645992535 to 573.6327437537788
Capped margin range: -335.74670645992535 to 537.1648449664455


In [18]:
worst_row = df_sales.loc[df_sales['Profit_Margin_Pct_Capped'].idxmin()]
print(worst_row[['Sales', 'Sales_Capped', 'Profit', 'Profit_Capped', 'Profit_Margin_Pct_Capped']])

Sales                          11.9136
Sales_Capped                   11.9136
Profit                      -39.999518
Profit_Capped               -39.999518
Profit_Margin_Pct_Capped   -335.746706
Name: 6559, dtype: object


In [19]:
outliers_margin, lb_m, ub_m = detect_outliers_iqr(df_sales, 'Profit_Margin_Pct_Capped')
df_sales['Profit_Margin_Pct_Final'] = df_sales['Profit_Margin_Pct_Capped'].clip(lower=lb_m, upper=ub_m)

print(f"Acceptable margin range: {lb_m:.2f}% to {ub_m:.2f}%")
print("Final margin range:", df_sales['Profit_Margin_Pct_Final'].min(), "to", df_sales['Profit_Margin_Pct_Final'].max())

Acceptable margin range: -71.09% to 117.11%
Final margin range: -71.08875635640835 to 117.1084519751943


In [20]:
df_sales.to_csv('../data/processed/cleaned_retail_sales.csv', index=False)
print("Saved:", df_sales.shape)

Saved: (10000, 30)


In [21]:
import os
print(os.path.exists('../data/processed/cleaned_retail_sales.csv'))
print(os.path.getsize('../data/processed/cleaned_retail_sales.csv'), "bytes")

True
3420825 bytes
